# Project 2: Supervised Learning — Fraud Detection Pipeline

## Project Goal
Build and tune classification models to identify fraudulent transactions in a highly imbalanced dataset.

## Key Objectives

1. **Handle Class Imbalance**
   - Applied **SMOTE (Synthetic Minority Over-sampling Technique)** to generate synthetic minority-class samples rather than duplicate existing records.

2. **Leak-Free Architecture**
   - Used `imblearn.pipeline.Pipeline` to ensure scaling and SMOTE were applied only within training folds during cross-validation.
   - Followed a **Zero-Leakage Protocol** to prevent information from the test set influencing model training.

3. **Model Evaluation**
   - Evaluated models using **Precision, Recall, F1-Score, and ROC-AUC** rather than relying on accuracy.
   - These metrics provide better insight into fraud detection performance under class imbalance.

4. **Model Comparison & Tuning**
   - Compared **Logistic Regression** and **Random Forest**.
   - Used `GridSearchCV` with cross-validation to tune model hyperparameters.

## Key Takeaway
This project demonstrates practical experience with **imbalanced classification, SMOTE, leak-free pipelines, cross-validation, hyperparameter tuning, and fraud-focused model evaluation**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Model Selection & Metrics
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve

# Preprocessing & Models
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Imbalanced Learn Pipeline & SMOTE (Prevents Data Leakage)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("Environment successfully configured!")

Environment successfully configured!


In [19]:
# 1. Load the pre-cleaned dataset exported from Project 1
df_cleaned = pd.read_csv("cleaned_production_dataset.csv")
print("Columns in cleaned_project1_dataset.csv:", df.columns.tolist())

Columns in cleaned_project1_dataset.csv: ['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice', 'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber', 'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice', 'TotalPrice_Capped', 'Is_Fraud', 'Effective_Item_Price', 'Is_High_Value_Order', 'Has_Discount']


In [24]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load Raw Dataset
df = pd.read_csv("Dataset for Data Analytics - Sheet1.csv")

# 2. Project 1 Cleaning: Impute Missing Coupon Codes
df["CouponCode"] = df["CouponCode"].fillna("NO_COUPON")

# 3. Project 1 Outlier Neutralization: Winsorize TotalPrice via IQR
Q1 = df["TotalPrice"].quantile(0.25)
Q3 = df["TotalPrice"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
df["TotalPrice_Capped"] = np.clip(
    df["TotalPrice"], a_min=None, a_max=upper_bound
)

# 4. Project 1 Feature Engineering: Generate 3 Metrics
df["Effective_Item_Price"] = df["TotalPrice_Capped"] / df["ItemsInCart"]
df["Is_High_Value_Order"] = (df["TotalPrice_Capped"] > Q3).astype(int)
df["Has_Discount"] = (df["CouponCode"] != "NO_COUPON").astype(int)

# 5. Project 2 Fraud Target Formulation
# Define fraud on the raw text OrderStatus before encoding
df["Is_Fraud"] = (
    (df["OrderStatus"].isin(["Cancelled", "Returned"]))
    & (df["TotalPrice_Capped"] > Q3)
).astype(int)

# 6. One-Hot Encoding
feature_cols = [
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice_Capped",
    "Effective_Item_Price",
    "Is_High_Value_Order",
    "Has_Discount",
    "Product",
    "PaymentMethod",
    "ReferralSource",
    "CouponCode",
]

X = pd.get_dummies(df[feature_cols], drop_first=True)
y = df["Is_Fraud"]

# 7. Stratified Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 8. Scale Features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 9. Train Balanced Logistic Regression Model
lr_model = LogisticRegression(
    class_weight="balanced", random_state=42, max_iter=1000
)
lr_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred_lr = lr_model.predict(X_test_scaled)
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print(" Execution Successful!")
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_proba_lr), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print(
    "\nClassification Report:\n",
    classification_report(y_test, y_pred_lr, digits=4),
)

 Execution Successful!
ROC-AUC Score: 0.9267
Confusion Matrix:
 [[184  31]
 [  0  25]]

Classification Report:
               precision    recall  f1-score   support

           0     1.0000    0.8558    0.9223       215
           1     0.4464    1.0000    0.6173        25

    accuracy                         0.8708       240
   macro avg     0.7232    0.9279    0.7698       240
weighted avg     0.9423    0.8708    0.8905       240



In [25]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Configure Stratified 5-Fold Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Define Parameter Grid
param_grid_lr = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'solver': ['liblinear', 'lbfgs']
}

# 3. Fit Grid Search using ROC-AUC as the Optimization Metric
grid_lr = GridSearchCV(
    estimator=LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    param_grid=param_grid_lr,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1
)

grid_lr.fit(X_train_scaled, y_train)

# 4. Predict on Untouched Test Set using Best Estimator
best_lr = grid_lr.best_estimator_
y_pred_best = best_lr.predict(X_test_scaled)
y_proba_best = best_lr.predict_proba(X_test_scaled)[:, 1]